In [5]:
import pandas as pd
import numpy as np
import os, zipfile
from lightgbm import LGBMRegressor


In [8]:
import os
print(os.getcwd())
print(os.listdir())

/content
['.config', 'sample_data']


In [25]:
train = pd.read_excel("train_target.xlsx")


def preprocess(df):
    df = df.copy()

    if df['시점'].astype(str).str.contains('T-').any():
        # test 데이터
        df['year'] = 0
        df['month'] = 0
        df['순'] = df['시점'].str.extract('(상순|중순|하순)')
    else:
        # train 데이터
        df['year'] = df['시점'].astype(str).str[:4].astype(int)
        df['month'] = df['시점'].astype(str).str[4:6].astype(int)
        df['순'] = df['시점'].astype(str).str[6:]


    mapping = {'상순': 0, '중순': 1, '하순': 2}
    df['순_num'] = df['순'].map(mapping)

    def get_season(m):
        if m in [3,4,5]: return 0
        elif m in [6,7,8]: return 1
        elif m in [9,10,11]: return 2
        else: return 3

    df['season'] = df['month'].apply(get_season)

    group_cols = ['품목명','품종명','거래단위','등급']
    df = df.sort_values(group_cols + ['year','month','순_num'])

    for lag in [1,2,3]:
        df[f'lag_{lag}'] = df.groupby(group_cols)['평균가격(원)'].shift(lag)

    df['rolling_mean_3'] = df.groupby(group_cols)['평균가격(원)'].transform(lambda x: x.shift(1).rolling(3).mean())
    df['rolling_std_3'] = df.groupby(group_cols)['평균가격(원)'].transform(lambda x: x.shift(1).rolling(3).std())

    df['pct_change'] = df.groupby(group_cols)['평균가격(원)'].pct_change()

    df['group_mean'] = df.groupby(group_cols)['평균가격(원)'].transform('mean')
    df['price_vs_avg'] = df['평균가격(원)'] / df['group_mean']
    df['diff_avg'] = df['평균가격(원)'] - df['group_mean']

    df['log_wholesale_vol'] = np.log1p(df['도매_총반입량'])
    df['vol_lag_1'] = df.groupby(group_cols)['도매_총반입량'].shift(1)
    df['wholesale_vol_mean_3'] = df.groupby(group_cols)['도매_총반입량'].transform(lambda x: x.shift(1).rolling(3).mean())

    df['price_per_vol'] = df['평균가격(원)'] / (df['도매_총반입량'] + 1)

    df['김장철'] = df['month'].isin([11,12]).astype(int)
    df['추석'] = df['month'].isin([9]).astype(int)

    return df

train = preprocess(train)

train = train.fillna(method='ffill').fillna(method='bfill')

/tmp/ipykernel_1627/3044997789.py:58: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  train = train.fillna(method='ffill').fillna(method='bfill')


In [12]:
import glob

test_files = sorted(glob.glob("/TEST_*_with_volume.csv"))
print(len(test_files))  # 25개 확인

25


In [27]:
features = [
    'month','순_num','season',
    'lag_1','lag_2','lag_3',
    'rolling_mean_3','rolling_std_3',
    'pct_change',
    'price_vs_avg','diff_avg',
    'vol_lag_1','log_wholesale_vol',
    'wholesale_vol_mean_3','price_per_vol',
    '김장철','추석'
]

target = '평균가격(원)'

X_train = train[features]
y_train = train[target]

In [26]:
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor

lgb = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

lgb.fit(X_train, y_train)
rf.fit(X_train, y_train)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000297 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3084
[LightGBM] [Info] Number of data points in the train set: 1440, number of used features: 17
[LightGBM] [Info] Start training from score 84185.646858


RandomForestRegressor(max_depth=10, n_estimators=300, n_jobs=-1,
                      random_state=42)

In [36]:
def predict_one(test_path):
    test = pd.read_csv(test_path)

    temp = pd.concat([train, test])
    temp = preprocess(temp)

    print("inf 개수:", np.isinf(temp[features]).sum())

    temp = temp.replace([np.inf, -np.inf], np.nan)
    temp = temp.fillna(method='ffill').fillna(method='bfill')

    temp_test = temp.iloc[-len(test):]
    X_test = temp_test[features]

    pred = 0.7*lgb.predict(X_test) + 0.3*rf.predict(X_test)

    result = test[['시점','품목명']].copy()
    result['평균가격(원)'] = pred

    return result

In [37]:
pred_dfs = []

for path in test_files:
    pred_df = predict_one(path)
    pred_dfs.append(pred_df)

submission = pd.concat(pred_dfs, ignore_index=True)

inf 개수: month                    0
순_num                    0
season                   0
lag_1                    0
lag_2                    0
lag_3                    0
rolling_mean_3           0
rolling_std_3            0
pct_change              14
price_vs_avg             0
diff_avg                 0
vol_lag_1                0
log_wholesale_vol        0
wholesale_vol_mean_3     0
price_per_vol            0
김장철                      0
추석                       0
dtype: int64


/tmp/ipykernel_1627/1351517606.py:10: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  temp = temp.fillna(method='ffill').fillna(method='bfill')


inf 개수: month                    0
순_num                    0
season                   0
lag_1                    0
lag_2                    0
lag_3                    0
rolling_mean_3           0
rolling_std_3            0
pct_change              31
price_vs_avg             0
diff_avg                 0
vol_lag_1                0
log_wholesale_vol        0
wholesale_vol_mean_3     0
price_per_vol            0
김장철                      0
추석                       0
dtype: int64


/tmp/ipykernel_1627/1351517606.py:10: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  temp = temp.fillna(method='ffill').fillna(method='bfill')


inf 개수: month                    0
순_num                    0
season                   0
lag_1                    0
lag_2                    0
lag_3                    0
rolling_mean_3           0
rolling_std_3            0
pct_change              24
price_vs_avg             0
diff_avg                 0
vol_lag_1                0
log_wholesale_vol        0
wholesale_vol_mean_3     0
price_per_vol            0
김장철                      0
추석                       0
dtype: int64


...

/tmp/ipykernel_1627/1351517606.py:10: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  temp = temp.fillna(method='ffill').fillna(method='bfill')


inf 개수: month                    0
순_num                    0
season                   0
lag_1                    0
lag_2                    0
lag_3                    0
rolling_mean_3           0
rolling_std_3            0
pct_change              33
price_vs_avg             0
diff_avg                 0
vol_lag_1                0
log_wholesale_vol        0
wholesale_vol_mean_3     0
price_per_vol            0
김장철                      0
추석                       0
dtype: int64


/tmp/ipykernel_1627/1351517606.py:10: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  temp = temp.fillna(method='ffill').fillna(method='bfill')


inf 개수: month                    0
순_num                    0
season                   0
lag_1                    0
lag_2                    0
lag_3                    0
rolling_mean_3           0
rolling_std_3            0
pct_change              32
price_vs_avg             0
diff_avg                 0
vol_lag_1                0
log_wholesale_vol        0
wholesale_vol_mean_3     0
price_per_vol            0
김장철                      0
추석                       0
dtype: int64


/tmp/ipykernel_1627/1351517606.py:10: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  temp = temp.fillna(method='ffill').fillna(method='bfill')


In [39]:
priority_map = {
    '특': 0,
    '상': 1,
    '상품': 2
}

submission['grade_priority'] = submission['등급'].map(priority_map)
submission['grade_priority'] = submission['grade_priority'].fillna(3)

submission = submission.sort_values(
    ['시점', '품목명', 'grade_priority']
)

KeyError: '등급'